In [1]:
import torch
import torch.nn as nn

# Single
from mpramnist.Agarwal2025.dataset import AgarwalSingleDataset
from mpramnist.Agarwal2025.trainer import LitModel_AgarwalSingle

# Multi
from mpramnist.Agarwal2025.dataset import AgarwalMultiDataset
from mpramnist.Agarwal2025.trainer import LitModel_AgarwalMulti

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights
import mpramnist.transforms as t

from torch.utils.data import DataLoader

import lightning.pytorch as L

/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Constant variables for denoting the added flanks.

# required for each sequence. flanks from original study
constant_left_flank = AgarwalSingleDataset.CONSTANT_LEFT_FLANK  
constant_rigtht_flank = (AgarwalSingleDataset.CONSTANT_RIGHT_FLANK)  

# original flanks from human MPRAlegnet. Using for shifting augmentation
left_flank = AgarwalSingleDataset.LEFT_FLANK  
right_flank = AgarwalSingleDataset.RIGHT_FLANK

In [3]:
# preprocessing
# The transformations for the training data differ from those used during testing.
train_transform = t.Compose(
    [
        t.AddFlanks(constant_left_flank, constant_rigtht_flank),
        t.AddFlanks("", right_flank),  # for shifting augmentation
        t.RightCrop(230, 260),  # for shifting augmentation
        t.LeftCrop(230, 230),
        t.ReverseComplement(0.5),
        t.Seq2Tensor(),
    ]
)
test_transform = t.Compose(
    [
        t.AddFlanks(constant_left_flank, constant_rigtht_flank),
        t.ReverseComplement(0),
        t.Seq2Tensor(),
    ]
)

In [4]:
# Testing is performed by averaging predictions for the forward and reverse-complement sequences. Pearson correlation.
from torchmetrics import PearsonCorrCoef

forw_transform = t.Compose(
    [t.AddFlanks(constant_left_flank, constant_rigtht_flank), t.Seq2Tensor()]
)
rev_transform = t.Compose(
    [
        t.AddFlanks(constant_left_flank, constant_rigtht_flank),
        t.ReverseComplement(1),
        t.Seq2Tensor(),
    ]
)

def meaned_prediction(forw, rev, trainer, seq_model, name, out_channels):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef(num_outputs=out_channels)
    print(name, " Pearson correlation")

    return pears(mean_forw, targets)

# AgarwalSingle

In [ ]:
# Reading data example
Cell_Type = "HepG2" # or K562 or WTC11

train_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type, split="train", root="../data/", # transform=train_transform
)

train_dataset[0]

('CCTAACCCTAACCCTAACCCTAACCCTAACCCCTAACCCTAACCCTAACCCTAACCCTCGCGGTACCCTCAGCCGGCCCGCCCGCCCGGGTCTGACCTGAGGAGAACTGTGCTCCGCCTTCAGAGTACCACCGAAATCTGTGCAGAGGACAACGCAGCTCCGCCCTCGCGGTGCTCTCCGGGTCTGTGCTGAGGAGAACG',
 tensor(-0.6750))

In [ ]:
# Read the MPRAdata, preprocess them
Cell_Type = "HepG2" # or K562 or WTC11
train_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type, split="train", transform=train_transform, root="../data/",
)

val_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type, split="val", transform=test_transform, root="../data/",
)

test_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type, split="test", transform=test_transform, root="../data/",
)

# encapsulate data into dataloader form
train_loader = DataLoader(
    dataset=train_dataset, batch_size=1024, shuffle=True, num_workers=16
)

val_loader = DataLoader(
    dataset=val_dataset, batch_size=1024, shuffle=False, num_workers=16
)

test_loader = DataLoader(
    dataset=test_dataset, batch_size=1024, shuffle=False, num_workers=16
)

in_channels = len(train_dataset[0][0])
out_channels = 1

In [7]:
print(train_dataset)

Dataset AgarwalSingleDataset (MpraDaraset)
    Number of datapoints: 98336
    Root location: ../data/Agarwal
    Using split: [1, 2, 3, 4, 5, 6, 7, 8]
    Split: {'train': 98336, 'val': 12298, 'test': 12298}
    Task: Regression
    Description: The AgarwalSingle dataset is based on a lentiviral MPRA system. The total data volume was 122,926 sequences for the HepG2 cell line, 196,664 for K562, and 46,185 for WTC11. Each sequence is 200 nucleotides long. The regression task was to predict a scalar value of regulatory activity for the corresponding cell line.


In [8]:
# Initialize the model, set the parameters, and create the LitModel Trainer for training.
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_HepG2 = LitModel_AgarwalSingle(
    model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=10
)

In [9]:
# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=5, # 5 epochs for example. 35-50 epochs recommended
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [10]:
# Train the model
trainer.fit(seq_model_HepG2, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model_HepG2, dataloaders=test_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | train
1 | loss          | MSELoss         | 0      | train
2 | train_pearson | PearsonCorrCoef | 0      | train
3 | val_pearson   | PearsonCorrCoef | 0      | train
4 | test_pearson  | PearsonCorrCoef | 0      | train
----------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.290     Total estimated model params size (MB)
120       Modules in train mode
0         Modules in eval mode


Epoch 0: 100%|██████████| 97/97 [00:09<00:00, 10.13it/s, v_num=54, val_loss=3.430, val_pearson=0.432, train_loss=0.503]

Metric val_loss improved. New best score: 3.433


Epoch 1: 100%|██████████| 97/97 [00:12<00:00,  7.54it/s, v_num=54, val_loss=0.548, val_pearson=0.562, train_loss=0.409]

...

Metric val_loss improved by 0.014 >= min_delta = 0.0. New best score: 0.309
`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 97/97 [00:13<00:00,  7.27it/s, v_num=54, val_loss=0.309, val_pearson=0.699, train_loss=0.273]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 63.50it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.29875504970550537
      test_pearson          0.7132672071456909
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.29875504970550537, 'test_pearson': 0.7132672071456909}]

In [ ]:
# Prediction correlation.
test_forw = AgarwalSingleDataset(cell_type=Cell_Type, split="test", transform=forw_transform, root="../data/")
test_rev = AgarwalSingleDataset(cell_type=Cell_Type, split="test", transform=rev_transform, root="../data/")

forw_hepg2 = DataLoader(dataset=test_forw, batch_size=1024, shuffle=False, num_workers=16, pin_memory=True,)
rev_hepg2 = DataLoader(dataset=test_rev, batch_size=1024, shuffle=False, num_workers=16, pin_memory=True,)

meaned_prediction(forw_hepg2, rev_hepg2, trainer, seq_model_HepG2, "HepG2", out_channels)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 54.03it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 65.87it/s]
HepG2  Pearson correlation


tensor(0.7262)

# AgarwalMulti

In [ ]:
# Read the MPRAdata, preprocess them
cell_types = ["HepG2", "K562", "WTC11"]

train_dataset = AgarwalMultiDataset(
    cell_type=cell_types, split="train", transform=train_transform, root="../data/",
)

val_dataset = AgarwalMultiDataset(
    cell_type=cell_types, split="val", transform=test_transform, root="../data/",
)

test_dataset = AgarwalMultiDataset(
    cell_type=cell_types, split="test", transform=test_transform, root="../data/",
)

# encapsulate data into dataloader form
train_loader = DataLoader(
    dataset=train_dataset, batch_size=1024, shuffle=True, num_workers=16
)

val_loader = DataLoader(
    dataset=val_dataset, batch_size=1024, shuffle=False, num_workers=16
)

test_loader = DataLoader(
    dataset=test_dataset, batch_size=1024, shuffle=False, num_workers=16
)

in_channels = len(train_dataset[0][0])
out_channels = len(cell_types)

In [13]:
print(train_dataset)

Dataset AgarwalMultiDataset (MpraDaraset)
    Number of datapoints: 44264
    Root location: ../data/AgarwalJoint
    Using split: [1, 2, 3, 4, 5, 6, 7, 8]
    Split: {'train': 44264, 'val': 5541, 'test': 5541}
    Task: Regression
    Description: The AgarwalMulti (joint) dataset contains 55,338 sequences, each 200 nucleotides long, comprising enhancers from the HepG2, K562, and WTC11 cell lines. The regression task is to predict three scalar values of regulatory activity for each of these cell lines.


In [14]:
# Initialize the model, set the parameters, and create the LitModel Trainer for training.
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_multi = LitModel_AgarwalMulti(
    model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=10
)

In [15]:
# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=5, # 5 epochs for example. 35-50 epochs recommended
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [16]:
# Train the model
trainer.fit(seq_model_multi, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model_multi, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (44) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | train
1 | loss          | MSELoss         | 0      | train
2 | train_pearson | PearsonCorrCoef | 0      | train
3 | val_pearson   | PearsonCorrCoef | 0      | train
4 | test_pearson  | PearsonCorrCoef | 0      | train
----------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.292     Total estimated model params size (MB)
120       Modules in train m

Epoch 0: 100%|██████████| 44/44 [00:07<00:00,  6.26it/s, v_num=55, train_loss_step=0.720, val_loss=0.986, val_HepG2_pearson=0.481, val_K562_pearson=0.292, val_WTC11_pearson=0.444, val_pearson=0.406, train_loss_epoch=0.863]

Metric val_loss improved. New best score: 0.986


Epoch 1: 100%|██████████| 44/44 [00:11<00:00,  3.94it/s, v_num=55, train_loss_step=0.722, val_loss=0.932, val_HepG2_pearson=0.542, val_K562_pearson=0.407, val_WTC11_pearson=0.524, val_pearson=0.491, train_loss_epoch=0.722]

Metric val_loss improved by 0.054 >= min_delta = 0.0. New best score: 0.932


...

Metric val_loss improved by 0.096 >= min_delta = 0.0. New best score: 0.551
`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 44/44 [00:11<00:00,  3.86it/s, v_num=55, train_loss_step=0.461, val_loss=0.551, val_HepG2_pearson=0.694, val_K562_pearson=0.645, val_WTC11_pearson=0.668, val_pearson=0.669, train_loss_epoch=0.529]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 56.14it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   test_HepG2_pearson       0.6862159371376038
    test_K562_pearson       0.6315090656280518
   test_WTC11_pearson       0.6571224927902222
        test_loss           0.5766751766204834
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.5766751766204834,
  'test_HepG2_pearson': 0.6862159371376038,
  'test_K562_pearson': 0.6315090656280518,
  'test_WTC11_pearson': 0.6571224927902222}]

In [ ]:
# Prediction correlation.
test_forw = AgarwalMultiDataset(cell_type=cell_types, split="test", transform=forw_transform, root="../data/")
test_rev = AgarwalMultiDataset(cell_type=cell_types, split="test", transform=rev_transform, root="../data/")

forw_multi = DataLoader(dataset=test_forw, batch_size=1024, shuffle=False, num_workers=16, pin_memory=True,)
rev_multi = DataLoader(dataset=test_rev, batch_size=1024, shuffle=False, num_workers=16, pin_memory=True,)

meaned_prediction(forw_multi, rev_multi, trainer, seq_model_multi, cell_types, out_channels)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 65.48it/s]


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 64.84it/s]
['HepG2', 'K562', 'WTC11']  Pearson correlation


tensor([0.6998, 0.6447, 0.6669])